In [1]:
!pip install faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import os
import cv2
import json
import glob
from tqdm.notebook import tqdm

In [3]:
!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetB.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetB.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset/")

     67,949,686 100%   35.78MB/s    0:00:01 (xfr#1, to-chk=0/1)


In [4]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

#our ViT-S DINOv2, ImageNet weights are default
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14",pretrained=True)

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 160MB/s]


In [5]:
from typing import Optional, List
#assumes we're passed a train, val, or test set, and that all images are within class folders
def generate_dict_from_set(embeddings: Optional[List] = None, path_to_dataset = None):
  index = 0 if embeddings is None else int(embeddings[-1]['id']) + 1
  all_embeddings = [] if embeddings is None else embeddings
  for folder in os.listdir(path_to_dataset):
    label = folder
    fold_path = os.path.join(path_to_dataset, folder)
    for file in os.listdir(fold_path):
      filepath = os.path.join(fold_path, file)
      all_embeddings.append({'id': index, 'label':label, 'path': filepath})
      index += 1
  return all_embeddings

dataset = generate_dict_from_set(path_to_dataset="./dataset/train")

In [6]:
def get_label_path_index(dataset_entry, index):
  id = dataset_entry['id']
  id = np.array([id], dtype='int64') #needs to be a 1d nparray for faiss
  label = dataset_entry['label']
  path = dataset_entry['path']
  return id, label, path

In [7]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small", use_fast=True)

def load_image(img: str) -> torch.Tensor:
    """
    Load an image and return a tensor that can be used as an input to DINOv2.
    """
    img = Image.open(img)

    inputs = processor(images=img, return_tensors="pt")

    return inputs["pixel_values"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

In [8]:
import faiss
from faiss import normalize_L2
def create_index(files: list, output_path) -> faiss.IndexFlatIP:
    """
    Create an index that contains all of the images in the specified list of files.
    """
    #
    dimension = 384 #we already know DINOv2 outputs this dimension
    index = faiss.IndexFlatIP(dimension) #inner product index
    index = faiss.IndexIDMap(index)


    with torch.no_grad():
      for i, entry in enumerate(tqdm(files)):
        id, label, img_path = get_label_path_index(entry, i)

        embeddings = dinov2_vits14(load_image(img_path).to(device))

        embedding = embeddings[0].cpu().numpy()

        embedding = np.array(embedding).reshape(1, -1)

        #vectors need to be normalized both before adding to the index and before searching
        normalize_L2(embedding) #for the sake of using cosine similarity

        index.add_with_ids(embedding, id)

    with open(output_path + '.paths.json', 'w') as f:
      json.dump(files, f)

    faiss.write_index(index, OUTPUT_INDEX_PATH)
    print(f"Index created and saved to {output_path}")

    return index

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index = create_index(dataset, OUTPUT_INDEX_PATH)

  0%|          | 0/2434 [00:00<?, ?it/s]

Index created and saved to /content/vector.index


In [ ]:
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index.paths.json" "./"
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index" "./"

        185,667 100%  145.82MB/s    0:00:00 (xfr#1, to-chk=0/1)
      3,758,186 100%   16.37MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [13]:
!cp vector.index.paths.json /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/
!cp vector.index /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/

In [9]:
import faiss
import json

#https://towardsdatascience.com/building-an-image-similarity-search-engine-with-faiss-and-clip-2211126d08fa/ again
def load_faiss_index(index_path):
    index = faiss.read_index(index_path)
    with open(index_path + '.paths.json', 'r') as f:
        image_paths = json.load(f)
    print(f"Index loaded from {index_path}")
    return index, image_paths

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index, files = load_faiss_index(OUTPUT_INDEX_PATH)


Index loaded from /content/vector.index


In [10]:
from collections import Counter
import statistics
from faiss import normalize_L2

def majority_voting_cosine(faiss_index, embeddings, files):

  correct_guesses = []
  correct_distances = []
  incorrect_distances = []
  incorrect_guesses = []
  all_distances = []

  for i, entry in enumerate(tqdm(files)):
    id, label, img_path = get_label_path_index(entry, i)

    with torch.no_grad():
      query_vectors = dinov2_vits14(load_image(img_path).to(device))

      query_vector = query_vectors[0].cpu().numpy()

      query_vector = np.array(query_vector).reshape(1, -1)

      normalize_L2(query_vector) #have to normalize to do cosine similarity


    k = 5  # Number of nearest neighbors to retrieve
    distances, indices = faiss_index.search(query_vector, k)

    all_guesses = [] #all labels of nearest neighbors
    neighbor_distances = []

    for i, index in enumerate(indices[0]):
      #print(f"index: {index}")
      #print(f"files len: {len(files)}")
      distance = distances[0][i]

      #kind of a mess. getting the associated embeddings id/label with our neighbor index
      id = embeddings[index]['id']
      new_label = embeddings[index]['label']

      all_guesses.append(new_label)
      all_distances.append(distance)
      neighbor_distances.append(distance)
      print(f"Nearest neighbor {i+1}: {id}, {new_label} Distance {distance}")

    majority_vote = Counter(all_guesses)
    winner = sorted(all_guesses, key=lambda x: majority_vote[x], reverse=True)[0]
    if winner == label:
      print(f"most common label was {winner} which == original label {label}")
      correct_guesses.append(winner)
      correct_distances.extend(neighbor_distances)
    else:
      print(f"most common label was {winner} which != {label}")
      incorrect_guesses.append(winner)
      incorrect_distances.extend(neighbor_distances)

  print(f"Total accuracy: {len(correct_guesses)/(len(correct_guesses)+len(incorrect_guesses))}")

  print(f"Median of all distances: {statistics.median(all_distances)}")

  print(f"Median distance of incorrect guesses: {statistics.median(incorrect_distances)}")

  print(f"Median distance of correct guesses: {statistics.median(correct_distances)}")

  print(f"Lowest: {min(all_distances)} highest: {max(all_distances)}")



In [12]:
val_dataset = generate_dict_from_set(path_to_dataset="./dataset/val")
majority_voting_cosine(faiss_index, files, val_dataset)

  0%|          | 0/550 [00:00<?, ?it/s]

Nearest neighbor 1: 1860, L-XX Distance 0.9221011996269226
Nearest neighbor 2: 0, P-XX Distance 0.9084568619728088
Nearest neighbor 3: 5, P-XX Distance 0.9076516032218933
Nearest neighbor 4: 6, P-XX Distance 0.907016396522522
Nearest neighbor 5: 1796, L-XX Distance 0.9050424098968506
most common label was P-XX which == original label P-XX
Nearest neighbor 1: 0, P-XX Distance 0.9304468631744385
Nearest neighbor 2: 4, P-XX Distance 0.9284449815750122
Nearest neighbor 3: 3, P-XX Distance 0.9272338151931763
Nearest neighbor 4: 5, P-XX Distance 0.9272280931472778
Nearest neighbor 5: 1860, L-XX Distance 0.921693742275238
most common label was P-XX which == original label P-XX
Nearest neighbor 1: 4, P-XX Distance 0.9237089157104492
Nearest neighbor 2: 0, P-XX Distance 0.9191380143165588
Nearest neighbor 3: 6, P-XX Distance 0.90948486328125
Nearest neighbor 4: 1860, L-XX Distance 0.9060015082359314
Nearest neighbor 5: 1855, L-XX Distance 0.9037491083145142
most common label was P-XX which == o

In [18]:
!rsync -a --info=progress2 "drive/MyDrive/Datasets/masked-Dataset_C.zip" "./"
import zipfile
with zipfile.ZipFile("masked-Dataset_C.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-C/")

    101,083,341 100%   64.20MB/s    0:00:01 (xfr#1, to-chk=0/1)


In [19]:
def get_last_id(files):
  return files[-1]['id']

id = get_last_id(files) + 1 #we're starting from the end of the index
val_dataset = generate_dict_from_set(None, "./dataset-C/")
majority_voting_cosine(faiss_index, files, val_dataset)

  0%|          | 0/5218 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Nearest neighbor 4: 220, MO-R Distance 0.8122700452804565
Nearest neighbor 5: 21, L-MB Distance 0.8074018955230713
most common label was MO-R which != B-
Nearest neighbor 1: 163, MO-R Distance 0.8411703705787659
Nearest neighbor 2: 217, MO-R Distance 0.8375867009162903
Nearest neighbor 3: 373, MO-R Distance 0.8365166783332825
Nearest neighbor 4: 293, MO-R Distance 0.8308106064796448
Nearest neighbor 5: 187, MO-R Distance 0.8289569020271301
most common label was MO-R which != B-
Nearest neighbor 1: 106, MO-R Distance 0.7949750423431396
Nearest neighbor 2: 368, MO-R Distance 0.7935305833816528
Nearest neighbor 3: 2384, WX-P Distance 0.7813328504562378
Nearest neighbor 4: 1437, K-WP Distance 0.7794284224510193
Nearest neighbor 5: 1447, K-WP Distance 0.7785884141921997
most common label was MO-R which != B-
Nearest neighbor 1: 359, MO-R Distance 0.8234996199607849
Nearest neighbor 2: 269, MO-R Distance 0.818895161151886
Nearest neighbor 3: